In [1]:
import pandas as pd
import numpy as np
import nltk
import re
import json
import string
from collections import Counter
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

for pkg in ['punkt', 'stopwords', 'wordnet', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print('All libraries imported successfully!')

All libraries imported successfully!


In [2]:
#text cleaning func
def clean_text(text):
    if not isinstance(text, str):
        return []
    text   = text.lower()
    text   = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    return [
        lemmatizer.lemmatize(w)
        for w in tokens
        if w not in stop_words and len(w) > 2
    ]

print('clean_text() ready')

clean_text() ready


In [3]:
# Load the original dataset
df = pd.read_csv('Resume.csv')

print('Dataset loaded!')
print('Shape   :', df.shape)
print('Columns :', df.columns.tolist())
print()
print('Cleaning all resumes...')

# Apply cleaning to every resume
df['Cleaned_tokens'] = df['Resume_str'].apply(clean_text)
df['Cleaned_text']   = df['Cleaned_tokens'].apply(lambda t: ' '.join(t))
df['Token_count']    = df['Cleaned_tokens'].apply(len)

print(f'   Total resumes cleaned : {len(df)}')
print(f'   Average tokens/resume : {df["Token_count"].mean():.0f}')
print()
print('Resumes per category:')
print(df['Category'].value_counts().to_string())

Dataset loaded!
Shape   : (2484, 4)
Columns : ['ID', 'Resume_str', 'Resume_html', 'Category']

Cleaning all resumes... (takes ~45 seconds)
   Total resumes cleaned : 2484
   Average tokens/resume : 584

Resumes per category:
Category
INFORMATION-TECHNOLOGY    120
BUSINESS-DEVELOPMENT      120
ADVOCATE                  118
CHEF                      118
ENGINEERING               118
ACCOUNTANT                118
FINANCE                   118
FITNESS                   117
AVIATION                  117
SALES                     116
BANKING                   115
HEALTHCARE                115
CONSULTANT                115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22


In [9]:
#CELL 4: Hardcoded Skill Profiles for all 24 categories

category_keywords = {
    "INFORMATION-TECHNOLOGY": [
        "python", "java", "sql", "linux", "javascript", "html", "css",
        "network", "database", "software", "cloud", "aws", "azure",
        "docker", "kubernetes", "git", "agile", "scrum", "api", "rest",
        "machine", "learning", "tensorflow", "mysql", "mongodb", "oracle",
        "cisco", "security", "devops", "testing", "debugging", "scala",
        "hadoop", "spark", "tableau", "power", "excel", "windows", "server"
    ],
    "HR": [
        "recruitment", "payroll", "hris", "onboarding", "compliance",
        "labor", "staffing", "appraisal", "workforce", "grievance",
        "benefits", "compensation", "training", "development", "policy",
        "performance", "retention", "talent", "hiring", "interview",
        "orientation", "attendance", "employee", "relations", "engagement"
    ],
    "FINANCE": [
        "accounting", "audit", "budget", "tax", "portfolio", "revenue",
        "cpa", "ledger", "reconciliation", "forecasting", "equity",
        "variance", "cashflow", "financial", "analysis", "reporting",
        "gaap", "ifrs", "treasury", "investment", "banking", "credit",
        "risk", "compliance", "excel", "quickbooks", "sap", "erp"
    ],
    "ACCOUNTANT": [
        "accounting", "audit", "tax", "ledger", "reconciliation", "gaap",
        "cpa", "bookkeeping", "payroll", "budget", "financial", "reporting",
        "quickbooks", "sap", "excel", "tally", "invoice", "balance",
        "sheet", "profit", "loss", "compliance", "variance", "cost"
    ],
    "HEALTHCARE": [
        "patient", "clinical", "nursing", "medical", "hospital", "diagnosis",
        "treatment", "surgery", "pharmacy", "therapy", "ehr", "hipaa",
        "physician", "care", "health", "laboratory", "radiology", "icu",
        "pediatric", "cardiology", "emergency", "rehabilitation", "medication"
    ],
    "ENGINEER": [
        "mechanical", "electrical", "civil", "structural", "autocad",
        "solidworks", "matlab", "manufacturing", "quality", "design",
        "simulation", "testing", "maintenance", "project", "safety",
        "construction", "hydraulics", "thermodynamics", "robotics"
    ],
    "ENGINEERING": [
        "mechanical", "electrical", "civil", "structural", "autocad",
        "solidworks", "matlab", "manufacturing", "quality", "design",
        "simulation", "testing", "maintenance", "project", "safety",
        "construction", "hydraulics", "thermodynamics", "robotics"
    ],
    "CHEF": [
        "culinary", "kitchen", "menu", "cuisine", "pastry", "catering",
        "banquet", "sous", "baking", "recipe", "garnish", "plating",
        "grill", "inventory", "hygiene", "sanitation", "food", "cooking",
        "preparation", "nutrition", "hospitality", "restaurant", "buffet"
    ],
    "TEACHER": [
        "curriculum", "classroom", "lesson", "instruction", "student",
        "assessment", "pedagogy", "learning", "teaching", "education",
        "literacy", "numeracy", "grading", "mentoring", "counseling",
        "syllabus", "differentiation", "inclusion", "stem", "subject"
    ],
    "SALES": [
        "revenue", "target", "crm", "negotiation", "pipeline", "lead",
        "conversion", "prospecting", "quota", "retail", "b2b", "b2c",
        "account", "client", "territory", "forecasting", "closing",
        "upselling", "salesforce", "marketing", "product", "distribution"
    ],
    "BANKING": [
        "credit", "loan", "risk", "compliance", "kyc", "aml", "audit",
        "investment", "portfolio", "treasury", "forex", "trading",
        "banking", "financial", "analysis", "regulatory", "capital",
        "liquidity", "mortgage", "underwriting", "branch", "customer"
    ],
    "ADVOCATE": [
        "legal", "court", "litigation", "attorney", "counsel", "contract",
        "compliance", "regulatory", "drafting", "arbitration", "mediation",
        "corporate", "criminal", "civil", "intellectual", "property",
        "trademark", "patent", "due", "diligence", "brief", "hearing"
    ],
    "CONSULTANT": [
        "strategy", "analysis", "business", "process", "improvement",
        "stakeholder", "project", "management", "implementation", "change",
        "advisory", "research", "presentation", "excel", "powerpoint",
        "erp", "sap", "crm", "transformation", "optimization", "kpi"
    ],
    "DESIGNER": [
        "photoshop", "illustrator", "indesign", "figma", "sketch",
        "ux", "ui", "wireframe", "prototype", "typography", "branding",
        "logo", "color", "layout", "visual", "creative", "animation",
        "after", "effects", "premiere", "graphic", "web", "responsive"
    ],
    "DIGITAL-MEDIA": [
        "seo", "sem", "social", "media", "content", "marketing", "analytics",
        "google", "facebook", "instagram", "campaign", "email", "ppc",
        "wordpress", "blogging", "copywriting", "photoshop", "video",
        "youtube", "influencer", "strategy", "engagement", "branding"
    ],
    "PUBLIC-RELATIONS": [
        "media", "press", "release", "communication", "branding", "campaign",
        "crisis", "stakeholder", "journalism", "writing", "editing",
        "social", "event", "pitching", "networking", "strategy",
        "corporate", "government", "lobbying", "outreach", "spokesperson"
    ],
    "CONSTRUCTION": [
        "civil", "structural", "autocad", "blueprint", "estimation",
        "project", "safety", "osha", "concrete", "masonry", "plumbing",
        "electrical", "surveying", "scheduling", "procurement", "quality",
        "inspection", "contract", "foundation", "renovation", "sitework"
    ],
    "AVIATION": [
        "pilot", "aircraft", "navigation", "atc", "maintenance", "safety",
        "faa", "icao", "flight", "aeronautical", "hydraulics", "avionics",
        "turbine", "license", "instrument", "cabin", "crew", "scheduling",
        "logbook", "weather", "takeoff", "landing", "checklist"
    ],
    "AGRICULTURE": [
        "crop", "farming", "soil", "irrigation", "harvest", "pesticide",
        "fertilizer", "livestock", "agronomy", "horticulture", "seed",
        "greenhouse", "organic", "yield", "plantation", "drainage",
        "machinery", "veterinary", "food", "supply", "chain", "export"
    ],
    "AUTOMOBILE": [
        "engine", "transmission", "chassis", "diagnostics", "repair",
        "maintenance", "electrical", "hydraulics", "brakes", "suspension",
        "autocad", "manufacturing", "quality", "assembly", "testing",
        "obd", "hybrid", "fuel", "injection", "workshop", "service"
    ],
    "FITNESS": [
        "training", "nutrition", "exercise", "strength", "cardio",
        "rehabilitation", "anatomy", "physiology", "coaching", "yoga",
        "pilates", "weight", "loss", "program", "assessment", "wellness",
        "sports", "injury", "prevention", "certification", "gym", "diet"
    ],
    "ARTS": [
        "painting", "sculpture", "drawing", "illustration", "photography",
        "exhibition", "portfolio", "creative", "design", "animation",
        "fine", "arts", "digital", "art", "direction", "concept",
        "printmaking", "ceramics", "gallery", "curation", "visual"
    ],
    "APPAREL": [
        "fashion", "textile", "fabric", "garment", "sewing", "pattern",
        "merchandising", "retail", "buying", "sourcing", "production",
        "quality", "design", "trend", "color", "collection", "brand",
        "supply", "chain", "manufacturing", "export", "styling"
    ],
    "BPO": [
        "customer", "service", "call", "center", "support", "crm",
        "communication", "resolution", "escalation", "quality", "kpi",
        "inbound", "outbound", "chat", "email", "ticketing", "sla",
        "training", "retention", "billing", "technical", "helpdesk"
    ],
    "BUSINESS-DEVELOPMENT": [
        "strategy", "revenue", "growth", "market", "research", "lead",
        "generation", "partnership", "negotiation", "crm", "sales",
        "analysis", "pipeline", "forecasting", "b2b", "expansion",
        "client", "acquisition", "proposal", "presentation", "kpi"
    ]
}

# Save updated profiles
with open('category_keywords.json', 'w') as f:
    json.dump(category_keywords, f, indent=2)

print("Skill profiles loaded for all 24 categories")
print()
for cat in ['INFORMATION-TECHNOLOGY', 'HR', 'FINANCE', 'CHEF']:
    print(f"  {cat:<30} → {category_keywords[cat][:6]}")

Skill profiles loaded for all 24 categories

  INFORMATION-TECHNOLOGY         → ['python', 'java', 'sql', 'linux', 'javascript', 'html']
  HR                             → ['recruitment', 'payroll', 'hris', 'onboarding', 'compliance', 'labor']
  FINANCE                        → ['accounting', 'audit', 'budget', 'tax', 'portfolio', 'revenue']
  CHEF                           → ['culinary', 'kitchen', 'menu', 'cuisine', 'pastry', 'catering']


In [10]:
def extract_skills(tokens, category):
    if category not in category_keywords:
        return []
    skill_list = set(category_keywords[category])
    return list(set(tokens).intersection(skill_list))


def extract_experience(raw_text):
    if not isinstance(raw_text, str):
        return 0

    text = raw_text.lower()

    number_map = {
        'one':1, 'two':2, 'three':3, 'four':4, 'five':5,
        'six':6, 'seven':7, 'eight':8, 'nine':9, 'ten':10,
        'eleven':11, 'twelve':12, 'fifteen':15, 'twenty':20
    }

    patterns = [
        r'(\d+)\+?\s*years?\s*of\s*experience',
        r'(\d+)\+?\s*years?\s*experience',
        r'(\d+)\+?\s*yrs?\s*experience',
        r'(\d+)\+?\s*years?',
    ]

    found = []
    for pattern in patterns:
        matches = re.findall(pattern, text)
        found.extend([int(m) for m in matches if int(m) < 50])

    for word, val in number_map.items():
        if f'{word} year' in text or f'{word} yrs' in text:
            found.append(val)

    return max(found) if found else 0


# Apply experience detection to all resumes
df['Experience_years'] = df['Resume_str'].apply(extract_experience)

print('extract_skills() ready')
print('extract_experience() ready')
print(f'Experience extracted for all {len(df)} resumes')
print()
print('Experience statistics:')
print(df['Experience_years'].describe().round(1).to_string())

extract_skills() ready
extract_experience() ready
Experience extracted for all 2484 resumes

Experience statistics:
count    2484.0
mean        5.3
std         7.9
min         0.0
25%         0.0
50%         0.0
75%         9.2
max        44.0


In [11]:
def build_tfidf_matrix(filtered_df):
    """
    Builds a TF-IDF matrix from the cleaned resume texts.
    Each row = one resume. Each column = one word or phrase.
    Returns the fitted vectorizer and the matrix.
    """
    vectorizer = TfidfVectorizer(
        max_features = 5000,   # use top 5000 words by frequency
        ngram_range  = (1, 2), # include single words and two-word phrases
        min_df       = 2,      # ignore words appearing in fewer than 2 resumes
        sublinear_tf = True    # dampen effect of very frequent words
    )
    tfidf_matrix = vectorizer.fit_transform(filtered_df['Cleaned_text'])
    print(f'   TF-IDF matrix: {tfidf_matrix.shape[0]} resumes x {tfidf_matrix.shape[1]} terms')
    return vectorizer, tfidf_matrix


def score_resumes(jd_text, filtered_df, vectorizer, tfidf_matrix):

    jd_clean  = ' '.join(clean_text(jd_text))
    jd_vector = vectorizer.transform([jd_clean])
    scores    = cosine_similarity(jd_vector, tfidf_matrix)[0]
    result_df = filtered_df.copy()
    result_df['Similarity_score'] = scores
    return result_df


print('build_tfidf_matrix() ready')
print('score_resumes() ready')

build_tfidf_matrix() ready
score_resumes() ready


In [12]:
def rank_candidates(jd_text, category, top_n=10):
    filtered = df[df['Category'] == category].copy().reset_index(drop=True)
    if len(filtered) == 0:
        print(f'❌ No resumes found for category: {category}')
        return None

    print(f'   Screening {len(filtered)} resumes for: {category}')

    vec, matrix = build_tfidf_matrix(filtered)

    #Compute cosine similarity scores
    scored = score_resumes(jd_text, filtered, vec, matrix)

    # Extract skills for each resume
    scored['Skills_found'] = scored.apply(
        lambda row: extract_skills(row['Cleaned_tokens'], category), axis=1
    )

    # Skill match ratio (skills found / total skills in profile)
    total_skills          = len(category_keywords.get(category, []))
    scored['Skill_ratio'] = scored['Skills_found'].apply(
        lambda s: len(s) / total_skills if total_skills > 0 else 0
    )

    # Normalise experience score
    scored['Exp_score'] = scored['Experience_years'].apply(
        lambda y: min(y, 20) / 20
    )

    # Weighted final score out of 100
    scored['Final_score'] = (
        scored['Similarity_score'] * 0.60 +
        scored['Skill_ratio']      * 0.30 +
        scored['Exp_score']        * 0.10
    ) * 100

    # Sort by final score and take top N
    result = scored.sort_values('Final_score', ascending=False).head(top_n)

    # Format output table
    output = result[[
        'ID', 'Final_score', 'Similarity_score',
        'Skill_ratio', 'Experience_years', 'Skills_found'
    ]].copy()

    output['Final_score']      = output['Final_score'].round(2)
    output['Similarity_score'] = output['Similarity_score'].round(4)
    output['Skill_ratio']      = output['Skill_ratio'].round(4)
    output['Skills_found']     = output['Skills_found'].apply(
        lambda s: ', '.join(sorted(s)) if s else 'None'
    )

    output.insert(0, 'Rank', range(1, len(output) + 1))
    output = output.reset_index(drop=True)
    return output


print('rank_candidates() ready')
print()
print('All functions loaded')

rank_candidates() ready

All functions loaded


In [13]:
print('=' * 50)
print('      AI RESUME SCREENING SYSTEM')
print('=' * 50)

# Show all available categories
print('\nAvailable job categories:\n')
for i, cat in enumerate(sorted(df['Category'].unique()), 1):
    print(f'  {i:>2}. {cat}')

print()
user_category = input('Enter job category (exactly as shown above): ').strip().upper()

print('\nEnter job description (press Enter twice when done):')
lines = []
while True:
    line = input()
    if line == '':
        break
    lines.append(line)
user_jd = ' '.join(lines)

# Validate category
if user_category not in df['Category'].unique():
    print(f"\n❌ '{user_category}' not found. Please re-run and type exactly as shown.")
else:
    print(f'\nCategory  : {user_category}')
    print(f'JD length : {len(user_jd.split())} words')
    print('\nRunning screening...\n')

    results = rank_candidates(
        jd_text  = user_jd,
        category = user_category,
        top_n    = 10
    )

    print()
    print('=' * 50)
    print('        TOP 10 CANDIDATES')
    print('=' * 50)
    print()
    print(results.to_string(index=False))

    results.to_csv('screening_results.csv', index=False)
    print()
    print('Results saved to screening_results.csv')

      AI RESUME SCREENING SYSTEM

Available job categories:

   1. ACCOUNTANT
   2. ADVOCATE
   3. AGRICULTURE
   4. APPAREL
   5. ARTS
   6. AUTOMOBILE
   7. AVIATION
   8. BANKING
   9. BPO
  10. BUSINESS-DEVELOPMENT
  11. CHEF
  12. CONSTRUCTION
  13. CONSULTANT
  14. DESIGNER
  15. DIGITAL-MEDIA
  16. ENGINEERING
  17. FINANCE
  18. FITNESS
  19. HEALTHCARE
  20. HR
  21. INFORMATION-TECHNOLOGY
  22. PUBLIC-RELATIONS
  23. SALES
  24. TEACHER

Enter job category (exactly as shown above): INFORMATION-TECHNOLOGY

Enter job description (press Enter twice when done):
We are looking for a software developer with strong experience in Python and Java.


Category  : INFORMATION-TECHNOLOGY
JD length : 14 words

Running screening...

   Screening 120 resumes for: INFORMATION-TECHNOLOGY
   TF-IDF matrix: 120 resumes x 5000 terms

        TOP 10 CANDIDATES

 Rank       ID  Final_score  Similarity_score  Skill_ratio  Experience_years                                                              